In [42]:
# ===== IMPORTS =====
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from scipy.spatial import cKDTree


In [43]:
# ===== LOAD DATA =====
processed_df = pd.read_csv("processed_data.csv", low_memory=False)

In [44]:
# ===== SELECT GNN-RELEVANT COLUMNS =====
gnn_columns = [
    "Accident_Index", "Latitude", "Longitude",
    "1st_Road_Number", "2nd_Road_Number", "Speed_limit",
    "Year", "Month", "Day", "Hour", "Is_Weekend", "Is_Rush_Hour",
    "Is_Peak_Hour", "Is_Night", "Is_Morning", "Is_Evening",
    "Bad_Weather_Flag", "Poor_Visibility_Flag",
]

prefixes = [
    "Road_Type_", "1st_Road_Class_", "2nd_Road_Class_",
    "Junction_Control_", "Junction_Detail_", "Weather_Conditions_",
    "Road_Surface_Conditions_", "Light_Conditions_",
    "Urban_or_Rural_Area_", "Traffic_Density_Indicator_", "Season_"
]

for col in processed_df.columns:
    for prefix in prefixes:
        if col.startswith(prefix):
            if col not in gnn_columns:
                gnn_columns.append(col)
            break

gnn_df = processed_df[gnn_columns].copy()
gnn_df.to_csv("gnn_accident_dataset.csv", index=False)
print("gnn_df shape:", gnn_df.shape)


gnn_df shape: (2047244, 86)


In [45]:
# ===== FILTER TO JUNCTION ACCIDENTS =====
junction_col = "Junction_Detail_Not At Junction Or Within 20 Metres"
at_junction_df = gnn_df[gnn_df[junction_col] == 0].copy()
print("Accidents at junctions:", at_junction_df.shape[0])


Accidents at junctions: 1220029


In [46]:
# ===== GRID-SNAP COORDINATES INTO NODES =====
at_junction_df["node_lat"] = at_junction_df["Latitude"].round(3)
at_junction_df["node_lon"] = at_junction_df["Longitude"].round(3)
at_junction_df["node_id"] = (
    at_junction_df["node_lat"].astype(str) + "_" + at_junction_df["node_lon"].astype(str)
)

In [47]:
# ===== SCOPE TO 2017 =====
year_filtered_df = at_junction_df[at_junction_df["Year"] == 2017].copy()
print("Junction accidents in 2017:", year_filtered_df.shape[0])
print("Unique intersection nodes in 2017:", year_filtered_df["node_id"].nunique())


Junction accidents in 2017: 30015
Unique intersection nodes in 2017: 26582


In [48]:
# ===== BUILD NODE FEATURES =====
node_features = year_filtered_df.groupby("node_id").agg(
    latitude=("node_lat", "first"),
    longitude=("node_lon", "first"),
    avg_speed_limit=("Speed_limit", "mean"),
    accident_count=("Accident_Index", "count"),
    bad_weather_rate=("Bad_Weather_Flag", "mean"),
    poor_visibility_rate=("Poor_Visibility_Flag", "mean"),
).reset_index()

node_features["node_index"] = range(len(node_features))
print("node_features shape:", node_features.shape)


node_features shape: (26582, 8)


In [49]:
# ===== ASSIGN ROAD NUMBER TO EACH NODE =====
road_at_node = year_filtered_df.groupby("node_id")["1st_Road_Number"].agg(lambda x: x.mode()[0])
node_features["road_number"] = node_features["node_id"].map(road_at_node)


In [50]:
# ===== BUILD EDGES (nearest-neighbor ordering per road) =====
def order_nodes_by_nearest_neighbor(group_df):
    remaining = group_df.copy().reset_index(drop=True)
    start_row = remaining.sort_values(["latitude", "longitude"]).iloc[0]
    chain = [start_row["node_index"]]
    remaining = remaining[remaining["node_index"] != start_row["node_index"]]
    current = start_row

    while len(remaining) > 0:
        dists = np.sqrt(
            (remaining["latitude"] - current["latitude"])**2 +
            (remaining["longitude"] - current["longitude"])**2
        )
        nearest_pos = dists.idxmin()
        nearest_row = remaining.loc[nearest_pos]
        chain.append(nearest_row["node_index"])
        remaining = remaining.drop(nearest_pos)
        current = nearest_row

    return chain

edges = []

for road_num, group in node_features.groupby("road_number"):
    if road_num == 0:
        continue
    if len(group) < 2:
        continue
    ordered_nodes = order_nodes_by_nearest_neighbor(group)
    for i in range(len(ordered_nodes) - 1):
        edges.append((ordered_nodes[i], ordered_nodes[i + 1]))

print("Total edges created:", len(edges))

Total edges created: 13921


In [51]:
# ===== SORT NODE FEATURES BY INDEX =====
node_features_sorted = node_features.sort_values("node_index").reset_index(drop=True)


In [52]:
# ===== BUILD EDGE LABELS (nearest-segment accident matching) =====
edge_midpoints = []
for node_a, node_b in edges:
    lat_a = node_features_sorted.iloc[node_a]["latitude"]
    lon_a = node_features_sorted.iloc[node_a]["longitude"]
    lat_b = node_features_sorted.iloc[node_b]["latitude"]
    lon_b = node_features_sorted.iloc[node_b]["longitude"]
    edge_midpoints.append(((lat_a + lat_b) / 2, (lon_a + lon_b) / 2))

edge_midpoints = np.array(edge_midpoints)
edge_tree = cKDTree(edge_midpoints)

all_2017_df = gnn_df[gnn_df["Year"] == 2017].copy()
all_2017_df = all_2017_df.dropna(subset=["Latitude", "Longitude"])
accident_coords = all_2017_df[["Latitude", "Longitude"]].values

distances, nearest_edge_idx = edge_tree.query(accident_coords)
all_2017_df["nearest_edge"] = nearest_edge_idx

edge_accident_counts = all_2017_df.groupby("nearest_edge").size()

new_edge_labels = np.zeros(len(edges))
for edge_i, count in edge_accident_counts.items():
    new_edge_labels[edge_i] = count
    new_edge_labels = new_edge_labels / new_edge_labels.max()
print("New edge label range:", new_edge_labels.min(), "to", new_edge_labels.max())

New edge label range: 0.0 to 1.0


In [53]:
# ===== INSTALL / IMPORT PYTORCH GEOMETRIC =====
# !pip install torch torch_geometric
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

In [54]:
# ===== BUILD GRAPH OBJECT =====
feature_columns = ["latitude", "longitude", "avg_speed_limit", "accident_count",
                    "bad_weather_rate", "poor_visibility_rate"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(node_features_sorted[feature_columns])
X = torch.tensor(X_scaled, dtype=torch.float)

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

y_edge = torch.tensor(new_edge_labels, dtype=torch.float)
y_edge = torch.cat([y_edge, y_edge])

data = Data(x=X, edge_index=edge_index)
print(data)



Data(x=[26582, 6], edge_index=[2, 27842])


In [55]:
# ===== MODEL =====
class RoadRiskGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.edge_predictor = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.conv2(h, edge_index)
        h = F.relu(h)
        return h
        
    def predict_edges(self, node_embeddings, edge_index):
        source_nodes = node_embeddings[edge_index[0]]
        target_nodes = node_embeddings[edge_index[1]]
        combined = torch.cat([source_nodes, target_nodes], dim=1)
        risk_score = self.edge_predictor(combined)
        risk_score = torch.sigmoid(risk_score)
        return risk_score.squeeze()

In [56]:
# ===== TRAIN =====
num_edges = edge_index.shape[1]
all_edge_positions = list(range(num_edges))
train_positions, test_positions = train_test_split(all_edge_positions, test_size=0.2, random_state=42)

model = RoadRiskGNN(input_dim=X.shape[1], hidden_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_function = nn.MSELoss()

epochs = 400
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    node_embeddings = model(data.x, data.edge_index)
    all_predictions = model.predict_edges(node_embeddings, data.edge_index)
    train_predictions = all_predictions[train_positions]
    train_labels = y_edge[train_positions]
    loss = loss_function(train_predictions, train_labels)
    loss.backward()
    optimizer.step()
    if epoch % 40 == 0:
        print(f"Epoch {epoch}, Training Loss: {loss.item():.5f}")

Epoch 0, Training Loss: 0.26977
Epoch 40, Training Loss: 0.00009
Epoch 80, Training Loss: 0.00009
Epoch 120, Training Loss: 0.00009
Epoch 160, Training Loss: 0.00009
Epoch 200, Training Loss: 0.00009
Epoch 240, Training Loss: 0.00009
Epoch 280, Training Loss: 0.00009
Epoch 320, Training Loss: 0.00009
Epoch 360, Training Loss: 0.00009


In [57]:
# ===== EVALUATE =====
model.eval()
with torch.no_grad():
    node_embeddings = model(data.x, data.edge_index)
    all_predictions = model.predict_edges(node_embeddings, data.edge_index)

    test_predictions = all_predictions[test_positions]
    test_labels = y_edge[test_positions]
    test_loss = loss_function(test_predictions, test_labels)

    baseline_prediction = y_edge[train_positions].mean()
    baseline_test_predictions = torch.full_like(test_labels, baseline_prediction)
    baseline_loss = loss_function(baseline_test_predictions, test_labels)

print("Model test loss:   ", test_loss.item())
print("Baseline test loss:", baseline_loss.item())
print("Prediction spread (std dev):", test_predictions.std().item())
print("True label spread (std dev):", test_labels.std().item())

Model test loss:    1.2245962022205958e-09
Baseline test loss: 1.1794046628210708e-08
Prediction spread (std dev): 1.0381664651504252e-05
True label spread (std dev): 3.3290682040387765e-05


In [58]:
# ===== SAVE MODEL =====
torch.save(model.state_dict(), "gnn_model.pth")
print("Model saved.")

Model saved.


In [60]:
# ===== EXPORT PREDICTIONS =====
original_edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

model.eval()
with torch.no_grad():
    node_embeddings = model(data.x, data.edge_index)
    final_predictions = model.predict_edges(node_embeddings, original_edge_index)

export_records = []
for i in range(len(edges)):
    node_a, node_b = edges[i]
    start_lat = node_features_sorted.iloc[node_a]["latitude"]
    start_lon = node_features_sorted.iloc[node_a]["longitude"]
    end_lat = node_features_sorted.iloc[node_b]["latitude"]
    end_lon = node_features_sorted.iloc[node_b]["longitude"]
    road_number = node_features_sorted.iloc[node_a]["road_number"]
    predicted_risk = final_predictions[i].item()

    export_records.append({
        "edge_id": i,
        "road_number": road_number,
        "start_lat": start_lat, "start_lon": start_lon,
        "end_lat": end_lat, "end_lon": end_lon,
        "predicted_risk": round(predicted_risk, 4)
    })

with open("gnn_risk_predictions.json", "w") as f:
    json.dump(export_records, f, indent=2)

print("Total road segments exported:", len(export_records))
print("Exported to gnn_risk_predictions.json")


Total road segments exported: 13921
Exported to gnn_risk_predictions.json
